# Module 4: PGD Federated Attack


## Stage Goal

This notebook isolates targeted PGD poisoning. The malicious clients take repeated projected surrogate-gradient steps within a pixel-space L-infinity budget. Run `train_v3.ipynb` and `train_surrogate.ipynb` first so the target and surrogate checkpoints exist.


## 1. Notebook Setup


In [ ]:
from pathlib import Path
import sys

MODULE_DIR = Path.cwd()
if (MODULE_DIR / "4_Adversarial_FL").exists():
    MODULE_DIR = MODULE_DIR / "4_Adversarial_FL"
SRC_DIR = MODULE_DIR / "src"
for path in (MODULE_DIR.parent, SRC_DIR):
    path_text = str(path)
    if path_text not in sys.path:
        sys.path.insert(0, path_text)

from IPython.display import display
import matplotlib.pyplot as plt

from attack_notebook_utils import (
    clean_baseline_table,
    clean_vs_attack_table,
    plot_algorithm_sweep,
    plot_clean_history,
    plot_clean_vs_attack,
    plot_sweep_table,
    prepare_context,
    run_algorithm_sweep,
    run_attack_parameter_sweep,
    run_basic_attack,
    run_clean_baseline,
)

plt.rcParams.update({"figure.dpi": 120, "axes.grid": False})


## 2. Configuration

Edit this cell to change data, FL, attack, or sweep settings. No YAML config is used.


In [ ]:
BASE_FED_CONFIG = {
    "fraction_clients": 0.2,
    "num_clients": 50,
    "num_rounds": 12,
    "num_epochs": 3,
    "batch_size": 64,
    "global_stepsize": 1.0,
    "local_stepsize": 0.002,
    "criterion": "torch.nn.CrossEntropyLoss",
}

ALGORITHMS = {
    "FedAvg": {
        "fed_config": {**BASE_FED_CONFIG, "global_stepsize": 1.0},
        "optim_config": {},
    },
    "FedAdam": {
        "fed_config": {**BASE_FED_CONFIG, "global_stepsize": 0.1},
        "optim_config": {"beta1": 0.9, "beta2": 0.99, "epsilon": 1e-6},
    },
    "FedAdagrad": {
        "fed_config": {**BASE_FED_CONFIG, "global_stepsize": 0.1},
        "optim_config": {"epsilon": 1e-6},
    },
    "FedYogi": {
        "fed_config": {**BASE_FED_CONFIG, "global_stepsize": 0.1},
        "optim_config": {"beta1": 0.9, "beta2": 0.99, "epsilon": 1e-6},
    },
    "Scaffold": {
        "fed_config": {**BASE_FED_CONFIG, "global_stepsize": 0.1},
        "optim_config": {"c_init": 0.0},
    },
}

ATTACK_NAME = "PGD"

BASE_ATTACK = {
    "type": "pgd",
    "targeted": True,
    "target_label": 0,
    "poison_rate": 0.2,
    "epsilon": 8 / 255,
    "step_size": 2 / 255,
    "iters": 20,
    "criterion": "torch.nn.CrossEntropyLoss",
}

PARAMETER_SWEEP = [
    {"name": "PGD 4/255 x10", "attack": {"epsilon": 4 / 255, "step_size": 1 / 255, "iters": 10}},
    {"name": "PGD 8/255 x10", "attack": {"epsilon": 8 / 255, "step_size": 2 / 255, "iters": 10}},
    {"name": "PGD 8/255 x20", "attack": {"epsilon": 8 / 255, "step_size": 2 / 255, "iters": 20}},
    {"name": "PGD 12/255 x20", "attack": {"epsilon": 12 / 255, "step_size": 3 / 255, "iters": 20}},
]

CONFIG = {
    "attack_name": ATTACK_NAME,
    "quiet": True,
    "data_config": {
        "dataset_path": "./Data/Imagenette",
        "dataset_name": "Imagenette",
        "non_iid_per": 0,
        "eval_subset": "attack_eval",
        "validation_split": {"enabled": True, "seed": 42, "selection_fraction": 0.5},
    },
    "global_config": {"seed": 42, "device": "cuda"},
    "artifacts": {
        "dir": "artifacts",
        "target_checkpoint": "module4_v3_target.pt",
        "surrogate_checkpoint": "module4_surrogate.pt",
    },
    "model_config": {
        "module": "model",
        "name": "MobileNetV3Transfer",
        "kwargs": {"pretrained": False, "num_classes": 10, "dropout": 0.1},
    },
    "algorithms": ALGORITHMS,
    "attack": {
        "seed": 42,
        "malicious_fraction": 0.1,
        "malicious_client_selection": {"mode": "seeded_random", "client_ids": []},
        "start_round": 3,
        "attack": BASE_ATTACK,
        "surrogate": {
            "checkpoint": "artifacts/module4_surrogate.pt",
            "pretrained": False,
            "num_classes": 10,
            "finetune_epochs": 0,
            "batch_size": 64,
            "learning_rate": 0.001,
            "weight_decay": 0.0,
            "freeze_backbone": False,
            "early_stop_patience": 0,
        },
    },
    "parameter_sweep": PARAMETER_SWEEP,
    "algorithm_sweep": ["FedAvg", "FedAdam", "FedAdagrad", "FedYogi", "Scaffold"],
}

context = prepare_context(CONFIG)
print(f"Device: {context['global_config']['device']}")
print(f"Target checkpoint: {context['target_checkpoint']}")
print(f"Surrogate checkpoint: {context['surrogate_checkpoint']}")


## 3. Clean FedAvg Baseline

This starts FedAvg from the saved MobileNetV3 checkpoint with no malicious clients.


In [ ]:
clean_fedavg = run_clean_baseline(context, "FedAvg")

display(clean_baseline_table(clean_fedavg))
plot_clean_history(clean_fedavg, title="Clean FedAvg baseline")


## 4. Basic FedAvg Attack

This reruns FedAvg with the notebook attack enabled for the configured malicious-client fraction.


In [ ]:
fedavg_attack = run_basic_attack(context, "FedAvg")

display(clean_vs_attack_table(clean_fedavg, fedavg_attack))
plot_clean_vs_attack(
    clean_fedavg,
    fedavg_attack,
    title=f"FedAvg: clean vs {CONFIG['attack_name']}",
    attack_start_round=CONFIG["attack"]["start_round"],
)


## 5. Attack Parameter Sweep

This keeps FedAvg fixed and changes only the attack settings listed in `PARAMETER_SWEEP`.


In [ ]:
parameter_sweep = run_attack_parameter_sweep(context, clean_fedavg, "FedAvg")

display(parameter_sweep["table"])
plot_sweep_table(
    parameter_sweep["table"],
    title=f"FedAvg {CONFIG['attack_name']} parameter sweep",
)


## 6. Algorithm Sweep

This compares the configured FL algorithms under the same attack recipe.


In [ ]:
algorithm_sweep = run_algorithm_sweep(context, fedavg_clean_result=clean_fedavg)

display(algorithm_sweep["table"])
plot_algorithm_sweep(
    algorithm_sweep["table"],
    title=f"Algorithm sweep under {CONFIG['attack_name']}",
)


## Handoff

Use the final attacked accuracy, accuracy drop, surrogate poison success, and `global_target_label_asr` to compare attack strength and motivate Module 5 defenses.
